# Tutorial 08 — Dual-frequency non-Rayleigh snowfall spectra (Billault-Roux 2023)

Reference: Billault-Roux, A.-C., Ghiggi, G., Jaffeux, L., Martini, A.,
Viltard, N., and Berne, A., 2023. *Dual-frequency spectral radar
retrieval of snowfall microphysics: a physics-driven deep-learning
approach*, Atmos. Meas. Tech., 16, 911–931
(doi:10.5194/amt-16-911-2023).

The paper relies on a simple but powerful asymmetry: at cloud-radar
frequencies small, slow-falling snow particles are still Rayleigh, so
their X-band and W-band reflectivities agree — but the large,
fast-falling ones are non-Rayleigh at W-band, so their W-band
spectral reflectivity is *smaller* than X-band. The **spectral
dual-wavelength ratio** sDWR(v) = 10·log₁₀(sZ_X / sZ_W) stays near 0
at low velocities and rises to several dB at high velocities — a
direct fingerprint of the large-particle size distribution tail,
untangled from turbulence and wind offsets.

This notebook reproduces that signature by running the same snow
scatterer, PSD, fall-speed, and turbulence through
`SpectralIntegrator` at X-band (9.5 GHz) and W-band (94 GHz).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rustmatrix import Scatterer, SpectralIntegrator, radar, spectra
from rustmatrix.psd import ExponentialPSD, PSDIntegrator
from rustmatrix.refractive import mi
from rustmatrix.tmatrix_aux import (K_w_sqr, geom_vert_back,
                                      wl_X, wl_W)


## Build matched snow scatterers at X-band and W-band

Both scatterers share the same oblate low-density aggregate habit,
PSD, and velocity grid. The *only* change across the two runs is the
radar wavelength.

PSD and habit parameters are grounded in the ICE-GENESIS 23 January
2021 case of Billault-Roux et al. 2023 (Fig. 5 snowfall layer):

* oblate aggregates, ρ_ice = 0.1 g/cm³ (low-density, typical of
  mixed-habit aggregates), axis ratio 0.6, D_max = 5 mm
* exponential PSD, N₀ = 2×10⁴ m⁻³ mm⁻¹, Λ = 2.5 mm⁻¹
* median-volume diameter D₀ = 3.67/Λ ≈ 1.5 mm
* IWC = (π ρ_ice / Λ⁴) · N₀ ≈ 0.16 g/m³ — a moderate aggregation
  layer, well within the synthetic training range the authors drew
  from their MASCDB disdrometer database.

These numbers land bulk Z_h(X) around 17 dBZ (moderate snowfall,
consistent with the paper's Fig. 5 observations at ~15:10 UTC) and a
non-Rayleigh-driven bulk DWR on the order of 15–20 dB — *not* the
50 dBZ / 25 dB outlier you get if you blindly pick Λ = 0.8 mm⁻¹ and
D_max = 10 mm (D₀ ≈ 5 mm, IWC > 10 g/m³ — a reflectivity-saturating
deep convective snow cell, not an aggregation layer).


In [ ]:
RHO_ICE = 0.1
AXIS_RATIO = 0.6
D_MAX = 5.0
N0 = 2e4
LAMBDA = 2.5

def build_snow(wl):
    s = Scatterer(wavelength=wl, m=mi(wl, RHO_ICE),
                  Kw_sqr=K_w_sqr[wl], axis_ratio=AXIS_RATIO,
                  ddelt=1e-4, ndgs=2)
    integ = PSDIntegrator()
    integ.D_max = D_MAX
    integ.num_points = 256
    integ.geometries = (geom_vert_back,)
    s.psd_integrator = integ
    s.psd_integrator.init_scatter_table(s)
    s.psd = ExponentialPSD(N0=N0, Lambda=LAMBDA, D_max=D_MAX)
    return s

snow_X = build_snow(wl_X)
snow_W = build_snow(wl_W)


## σ_b(D) at X-band vs W-band — the non-Rayleigh onset

Up to about 2 mm equivalent diameter the two σ_b(D) curves are
essentially parallel D⁶ power laws (Rayleigh). Above ~3 mm the
W-band curve rolls over and develops Mie structure while the X-band
curve keeps rising — this is the imprint the dual-frequency spectrum
will show up as a velocity-resolved sDWR.


In [ ]:
def sigma_b(wl, D_mm):
    s = Scatterer(radius=D_mm/2, wavelength=wl,
                  m=mi(wl, RHO_ICE), Kw_sqr=K_w_sqr[wl],
                  axis_ratio=AXIS_RATIO, ddelt=1e-4, ndgs=2)
    s.set_geometry(geom_vert_back)
    return radar.radar_xsect(s)

D = np.linspace(0.1, D_MAX, 150)
sb_X = np.array([sigma_b(wl_X, d) for d in D])
sb_W = np.array([sigma_b(wl_W, d) for d in D])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(D, sb_X, 'C0-', label='X-band (9 GHz)')
ax.loglog(D, sb_W, 'C3-', label='W-band (94 GHz)')
ax.set_xlabel('equivalent diameter D [mm]')
ax.set_ylabel(r'$\sigma_b$ [mm²]')
ax.set_title('Snow backscatter at vertical incidence')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout();


## Dual-frequency Doppler spectra

Same fall-speed model (Locatelli–Hobbs aggregates), same turbulence,
same velocity grid — only the radar wavelength changes.


In [ ]:
fall = spectra.fall_speed.locatelli_hobbs_1974_aggregates
turb = spectra.GaussianTurbulence(0.2)
V_MIN, V_MAX = -2.0, 4.0
N_BINS = 1024

def run(sc):
    return SpectralIntegrator(
        sc, fall_speed=fall, turbulence=turb,
        v_min=V_MIN, v_max=V_MAX, n_bins=N_BINS,
        geometry_backscatter=geom_vert_back,
    ).run()

r_X = run(snow_X)
r_W = run(snow_W)

def dBZ(x):
    return 10 * np.log10(np.maximum(x, 1e-12))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(r_X.v, dBZ(r_X.sZ_h), 'C0-', label='X-band')
ax1.plot(r_W.v, dBZ(r_W.sZ_h), 'C3-', label='W-band')
ax1.set_ylabel('sZ_h [dBZ / (m/s)]')
ax1.set_title('Dual-frequency spectra of snowfall — vertical pointing')
ax1.legend(); ax1.grid(True, alpha=0.3)

sDWR = dBZ(r_X.sZ_h) - dBZ(r_W.sZ_h)
good = (r_X.sZ_h > 1e-8) & (r_W.sZ_h > 1e-10)
ax2.plot(r_X.v[good], sDWR[good], 'C2-')
ax2.axhline(0.0, color='k', ls=':', alpha=0.6)
ax2.set_xlabel('Doppler velocity v [m/s]')
ax2.set_ylabel('sDWR$_{X-W}$ [dB]')
ax2.set_title('Spectral dual-wavelength ratio: Rayleigh at low v, '
              'non-Rayleigh at high v')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(V_MIN, V_MAX)
fig.tight_layout();


## Reading the signature

At velocities below ~0.5 m/s (small aggregates, well inside the
Rayleigh regime at both bands) sDWR is essentially zero. It rises
monotonically with v as the fastest-falling particles move into the
W-band non-Rayleigh regime while remaining Rayleigh at X-band. The
magnitude of the rise at a given velocity is a direct proxy for the
*size* of the drops there — which is exactly the lever Billault-Roux
et al. 2023 use to retrieve the PSD tail, with deep learning
shouldering the joint dependence on turbulence and shape.


In [ ]:
print(f'bulk Z_h (X-band) = '
      f'{10*np.log10(np.trapezoid(r_X.sZ_h, r_X.v)):.2f} dBZ')
print(f'bulk Z_h (W-band) = '
      f'{10*np.log10(np.trapezoid(r_W.sZ_h, r_W.v)):.2f} dBZ')
print(f'bulk DWR          = '
      f'{10*np.log10(np.trapezoid(r_X.sZ_h, r_X.v) / np.trapezoid(r_W.sZ_h, r_W.v)):.2f} dB')
print()
v_samples = [0.3, 0.5, 0.8, 1.0, 1.3, 1.6]
print('sDWR(v) at selected velocities:')
for vs in v_samples:
    i = int(np.argmin(np.abs(r_X.v - vs)))
    print(f'  v = {r_X.v[i]:.2f} m/s   sDWR = '
          f'{dBZ(r_X.sZ_h[i]) - dBZ(r_W.sZ_h[i]):+.2f} dB')
